# MuRIL Urgency Model

Trains a MuRIL classifier that predicts the **urgency level** (low / medium / high) of a complaint from its text.

**Run from the `notebooks/` folder** (the paths below go one level up to the project root).
It reuses the same `train.csv`, `validation.csv` and `test.csv` as the category model, so the test rows are identical for both models.

## Step 0: Install and Import Libraries

### 0.1 Install packages
Same packages as the category notebook, plus seaborn and matplotlib for the plots.

In [1]:
%pip install -q transformers datasets scikit-learn pandas openpyxl torch accelerate seaborn matplotlib

Note: you may need to restart the kernel to use updated packages.


### 0.2 Imports and seed
Fixes the random seed so results are reproducible, and prints whether a GPU is available.

In [2]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
print("GPU available:", torch.cuda.is_available())

/home/amrita/Projects/CivicAI-FinalYearProject/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU available: False


## Step 1: Set Paths and Load the Existing Splits

### 1.1 Paths
Points to the text dataset and to the folders where the model will be saved. Training checkpoints are kept separate from the final model so nothing from the category model is overwritten.

In [3]:
PROJECT_ROOT = os.path.abspath("..")   # notebook is in notebooks/; use "." if it is in the project root

TEXT_DIR = os.path.join(PROJECT_ROOT, "data", "text_dataset")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

CHECKPOINT_DIR = os.path.join(MODELS_DIR, "muril_urgency_checkpoints")
FINAL_DIR = os.path.join(MODELS_DIR, "muril_urgency_final")

print("Text data:", TEXT_DIR, "| exists:", os.path.isdir(TEXT_DIR))
print("Final model will be saved to:", FINAL_DIR)

Text data: /home/amrita/Projects/CivicAI-FinalYearProject/data/text_dataset | exists: True
Final model will be saved to: /home/amrita/Projects/CivicAI-FinalYearProject/models/muril_urgency_final


### 1.2 Load the same train/validation/test CSVs
Reuses the splits saved by the category notebook. Urgency labels are lower-cased and stripped to avoid mismatches like "High" vs "high".

In [4]:
train_df = pd.read_csv(os.path.join(TEXT_DIR, "train.csv"))
val_df = pd.read_csv(os.path.join(TEXT_DIR, "validation.csv"))
test_df = pd.read_csv(os.path.join(TEXT_DIR, "test.csv"))

for d in (train_df, val_df, test_df):
    d["urgency_level"] = d["urgency_level"].astype(str).str.lower().str.strip()

print("Train:", len(train_df), "Validation:", len(val_df), "Test:", len(test_df))
print("\nUrgency distribution per split (proportions):")
print(pd.DataFrame({
    "train": train_df["urgency_level"].value_counts(normalize=True),
    "validation": val_df["urgency_level"].value_counts(normalize=True),
    "test": test_df["urgency_level"].value_counts(normalize=True),
}).round(3))

Train: 3038 Validation: 652 Test: 652

Urgency distribution per split (proportions):
               train  validation   test
urgency_level                          
high           0.503       0.469  0.492
medium         0.247       0.248  0.248
low            0.130       0.149  0.143
critical       0.120       0.133  0.117


## Step 2: Understand the Urgency Labels

### 2.1 Score range per urgency level
Shows the lowest, average and highest `urgency_score` inside each level. Use these ranges to choose the priority-level thresholds in `config.py` instead of inventing numbers.

In [5]:
print(train_df.groupby("urgency_level")["urgency_score"].agg(["count", "min", "mean", "max"]).round(2))

               count  min  mean  max
urgency_level                       
critical         365  7.5  9.16  9.5
high            1529  5.2  7.67  9.8
low              394  2.0  3.16  6.0
medium           750  3.5  5.32  7.0


### 2.2 Shortcut check: how well do category and location_type alone predict urgency?
A simple lookup table (most common urgency for each category + location_type pair) is evaluated on the test set. If this trivial baseline is already very accurate, the urgency labels are largely rule-based, so a high MuRIL score says less about the model. Report this honestly in the limitations.

In [6]:
pair_lookup = train_df.groupby(["category", "location_type"])["urgency_level"].agg(lambda s: s.mode()[0])
cat_lookup = train_df.groupby("category")["urgency_level"].agg(lambda s: s.mode()[0])
global_major = train_df["urgency_level"].mode()[0]

def baseline_predict(row):
    key = (row["category"], row["location_type"])
    if key in pair_lookup.index:
        return pair_lookup[key]
    if row["category"] in cat_lookup.index:
        return cat_lookup[row["category"]]
    return global_major

baseline_preds = test_df.apply(baseline_predict, axis=1)
print("Rule-based baseline accuracy on test:", round(accuracy_score(test_df["urgency_level"], baseline_preds), 4))

Rule-based baseline accuracy on test: 0.4939


## Step 3: Encode Labels and Build the Model Input

### 3.1 Fixed label order and input text
Urgency is ordered, so labels are mapped explicitly (low=0, medium=1, high=2) rather than alphabetically.

`USE_LOCATION_TYPE` decides whether `location_type` is added in front of the description. Start with `False` (description only). If you set it to `True`, the app must also collect a location type. The setting is saved with the model so the engine uses the same input format.

In [7]:

import pandas as pd

# 1. Define the three urgency classes
LEVELS = ["low", "medium", "high"]

level2id = {level: i for i, level in enumerate(LEVELS)}
id2level = {i: level for level, i in level2id.items()}

USE_LOCATION_TYPE = False


# 2. Prepare the input text
def build_input(description, location_type=None):
    description = str(description).strip()

    if (
        USE_LOCATION_TYPE
        and isinstance(location_type, str)
        and location_type.strip()
    ):
        return f"{location_type.strip()}: {description}"

    return description


# 3. Process training, validation, and test datasets
for name, d in [
    ("train_df", train_df),
    ("val_df", val_df),
    ("test_df", test_df)
]:
    # Clean urgency labels
    d["urgency_level"] = (
        d["urgency_level"]
        .astype("string")
        .str.strip()
        .str.lower()
        .replace({"critical": "high"})
    )

    # Convert urgency levels into numeric labels
    d["label"] = d["urgency_level"].map(level2id)

    # Stop if any unexpected or missing labels remain
    if d["label"].isna().any():
        invalid = d.loc[
            d["label"].isna(), "urgency_level"
        ].unique().tolist()

        raise ValueError(
            f"Unexpected urgency levels in {name}: {invalid}"
        )

    d["label"] = d["label"].astype(int)

    # Handle missing descriptions safely
    d["description"] = d["description"].fillna("").astype(str)

    # Build model input text
    d["text"] = [
        build_input(description, location_type)
        for description, location_type in zip(
            d["description"], d["location_type"]
        )
    ]

    print(f"\n{name}:")
    print(f"Total rows: {len(d)}")
    print("Urgency distribution:")
    print(d["urgency_level"].value_counts())
    print("Numeric label distribution:")
    print(d["label"].value_counts().sort_index())


# 4. Display the training dataset preview
print("\nTraining dataset preview:")
print(
    train_df[["text", "urgency_level", "label"]].head()
)

# 5. Verify the label mapping
print("\nLabel mapping:")
print(level2id)

print("\nAll datasets processed successfully!")



train_df:
Total rows: 3038
Urgency distribution:
urgency_level
high      1894
medium     750
low        394
Name: count, dtype: int64[pyarrow]
Numeric label distribution:
label
0     394
1     750
2    1894
Name: count, dtype: int64

val_df:
Total rows: 652
Urgency distribution:
urgency_level
high      393
medium    162
low        97
Name: count, dtype: int64[pyarrow]
Numeric label distribution:
label
0     97
1    162
2    393
Name: count, dtype: int64

test_df:
Total rows: 652
Urgency distribution:
urgency_level
high      397
medium    162
low        93
Name: count, dtype: int64[pyarrow]
Numeric label distribution:
label
0     93
1    162
2    397
Name: count, dtype: int64

Training dataset preview:
                                                text urgency_level  label
0  Culvert filled with mud after rain, water is e...          high      2
1  फोहोर उठाउने गाडी ढिलो आउँदा व्यापारिक क्षेत्र...        medium      1
2  Community hall covered with political slogans ...          high

In [8]:
LEVELS = ["low", "medium", "high"]
level2id = {l: i for i, l in enumerate(LEVELS)}
id2level = {i: l for l, i in level2id.items()}

USE_LOCATION_TYPE = False

def build_input(description, location_type=None):
    if USE_LOCATION_TYPE and isinstance(location_type, str) and location_type.strip():
        return f"{location_type.strip()}: {description}"
    return description

for d in (train_df, val_df, test_df):
    d["label"] = d["urgency_level"].map(level2id)
    d["text"] = [build_input(a, b) for a, b in zip(d["description"], d["location_type"])]
    assert not d["label"].isna().any(), "Unexpected urgency_level value found"

print(train_df[["text", "urgency_level", "label"]].head())

                                                text urgency_level  label
0  Culvert filled with mud after rain, water is e...          high      2
1  फोहोर उठाउने गाडी ढिलो आउँदा व्यापारिक क्षेत्र...        medium      1
2  Community hall covered with political slogans ...          high      2
3  Sharp-bend sign hanging from one bolt in a pub...          high      2
4  Sharp-bend sign hidden by branches in Tulsipur...          high      2


## Step 4: Tokenize with the MuRIL Tokenizer

### 4.1 Tokenizer and Hugging Face datasets
Same tokenizer and max length as the category model. Each split becomes a Hugging Face `Dataset` in torch format.

In [9]:
MODEL_NAME = "google/muril-base-cased"
MAX_LENGTH = 128
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=MAX_LENGTH)

def to_dataset(df):
    ds = Dataset.from_pandas(df[["text", "label"]].reset_index(drop=True))
    ds = ds.map(tokenize_batch, batched=True)
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    return ds

train_ds = to_dataset(train_df)
val_ds = to_dataset(val_df)
test_ds = to_dataset(test_df)

Map: 100%|██████████| 652/652 [00:00<00:00, 3988.84 examples/s]


## Step 5: Load the MuRIL Model

### 5.1 Pretrained MuRIL with a 3-class head
Loads MuRIL with a fresh classification head for low / medium / high. `id2label` is stored in the model config so the saved model can always translate its output back to a level name.

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LEVELS),
    id2label=id2level,
    label2id=level2id,
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12832.34it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

## Step 6: Define Evaluation Metrics

### 6.1 Metrics function
Reports accuracy and macro-averaged precision, recall and F1, same as the category notebook. Macro-F1 is the main number because it gives each urgency level equal weight.

In [1]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

## Step 7: Train with Validation Monitoring

### 7.1 Training setup and run
Same recipe as the category model. `save_total_limit=1` keeps only one checkpoint on disk and `report_to="none"` turns off external logging prompts. The best epoch by validation F1 is reloaded at the end.

In [ ]:

import torch
from transformers import TrainingArguments, Trainer

# Reduce memory usage before training
model.config.use_cache = False

# Enable gradient checkpointing to save activation memory
model.gradient_checkpointing_enable()

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,

    # Evaluate and save once per epoch
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,

    # Lower memory usage
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,

    # Training configuration
    learning_rate=2e-5,
    num_train_epochs=2,
    weight_decay=0.01,

    # Select the best checkpoint using validation F1
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_steps=50,
    report_to="none",
    seed=SEED,

    # Use mixed precision only when CUDA is available
    fp16=torch.cuda.is_available(),

    # Reduce optimizer memory consumption
    optim="adafactor",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

/home/amrita/Projects/CivicAI-FinalYearProject/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'model' is not defined

## Step 8: Plot Training and Validation Curves

### 8.1 Loss and validation metrics per epoch
Plots training vs validation loss (overfitting check) and validation accuracy and F1 across epochs.

In [ ]:
history = trainer.state.log_history
train_logs = pd.DataFrame([h for h in history if "loss" in h and "eval_loss" not in h])
eval_logs = pd.DataFrame([h for h in history if "eval_loss" in h])

plt.figure(figsize=(8, 5))
plt.plot(train_logs["epoch"], train_logs["loss"], label="Training Loss", marker="o")
plt.plot(eval_logs["epoch"], eval_logs["eval_loss"], label="Validation Loss", marker="o")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Urgency Model: Training vs Validation Loss")
plt.legend(); plt.grid(True); plt.show()

plt.figure(figsize=(8, 5))
plt.plot(eval_logs["epoch"], eval_logs["eval_accuracy"], label="Validation Accuracy", marker="o")
plt.plot(eval_logs["epoch"], eval_logs["eval_f1"], label="Validation F1", marker="o")
plt.xlabel("Epoch"); plt.ylabel("Score"); plt.title("Urgency Model: Validation Accuracy and F1")
plt.legend(); plt.grid(True); plt.show()

## Step 9: Evaluate on the Held-out Test Set

### 9.1 Test metrics, per-class report and confusion matrix
Run once, at the end. The "within one level" number matters because urgency is ordered: predicting medium for a high complaint is a smaller mistake than predicting low for a high one.

In [ ]:
pred_out = trainer.predict(test_ds)
y_true = pred_out.label_ids
y_pred = np.argmax(pred_out.predictions, axis=-1)

print("Test accuracy: ", round(accuracy_score(y_true, y_pred), 4))
p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
print("Test precision:", round(p, 4), "| recall:", round(r, 4), "| macro-F1:", round(f1, 4))
print("Within one level:", round(float(np.mean(np.abs(y_true - y_pred) <= 1)), 4))
print("\n", classification_report(y_true, y_pred, target_names=LEVELS, digits=4))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LEVELS, yticklabels=LEVELS)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Urgency Confusion Matrix (Test)")
plt.show()

## Step 10: Save the Model, Score Statistics and Settings

### 10.1 Save everything the engine needs
Besides the model and tokenizer, this saves two small JSON files next to them:

- `urgency_stats.json` holds the average `urgency_score` per level (the priority base scores) and the typical urgency per category (used when a complaint has an image but no text).
- `urgency_config.json` records whether `location_type` was used.

All statistics come from the training split only.

In [ ]:
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

stats = {
    "level_base_score": {
        k: round(float(v), 2)
        for k, v in train_df.groupby("urgency_level")["urgency_score"].mean().items()
    },
    "category_default": {
        str(cat).lower(): {
            "level": g["urgency_level"].mode()[0],
            "score": round(float(g["urgency_score"].mean()), 2),
        }
        for cat, g in train_df.groupby("category")
    },
}
urgency_config = {"use_location_type": USE_LOCATION_TYPE, "max_length": MAX_LENGTH, "levels": LEVELS}

with open(os.path.join(FINAL_DIR, "urgency_stats.json"), "w", encoding="utf-8") as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)
with open(os.path.join(FINAL_DIR, "urgency_config.json"), "w", encoding="utf-8") as f:
    json.dump(urgency_config, f, indent=2)

print("Saved to", FINAL_DIR)
print(json.dumps(stats, indent=2, ensure_ascii=False))

## Step 11: Try Sample Complaints (Including Romanized Nepali)

### 11.1 Predict urgency for a few hand-written complaints
A quick look at the predicted probabilities for English, Devanagari and romanized Nepali examples. Weak or unsure romanized results here are worth noting for your report.

In [ ]:
def predict_level(description, location_type=None):
    text = build_input(description, location_type)
    inputs = tokenizer(text, return_tensors="pt", padding="max_length",
                       truncation=True, max_length=MAX_LENGTH).to(model.device)
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(**inputs).logits, dim=-1)[0].cpu().numpy()
    return {l: round(float(p), 3) for l, p in zip(LEVELS, probs)}

samples = [
    "road bigreko chha mhepi najikai",
    "ghar ko fohor sadak ma overflow bhayeko chha",
    "Public bin overflowing onto the pavement near the road junction.",
    "ढल बन्द भएर पानी सडकमा बगिरहेको छ",
]
for s in samples:
    print(s, "->", predict_level(s))